# Workflow 1: Flat React Agent - Ollama (Gemma 4 2B local)

In [1]:
# Tự động reload các module khi file .py thay đổi (tránh phải restart kernel)
%load_ext autoreload
%autoreload 2


In [2]:
import sys
import os
from pathlib import Path
# Thêm thư mục cha (rag-service) vào danh sách tìm kiếm của Python
notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)

import time
import re
import requests
import json
from typing import List, Dict, Any

import uuid
import operator
from typing import TypedDict, Literal, Optional, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

from configs.setting import settings
from configs.GetConfig import config

from src.LLMService import LLMService
from src.e_agents.guardrail_call import GuardrailCall
from src.e_agents.rejection_call import RejectionCall

from src.d_tools import (
    product_search, 
    product_compare,
    policy_search,
    order_lookup,
)

from src.d_tools import (
    PRODUCT_SEARCH_SCHEMA,
    PRODUCT_COMPARE_SCHEMA,
    POLICY_SEARCH_SCHEMA,
    ORDER_LOOKUP_SCHEMA
)

from src.f_prompts import (
    FULL_MASTER_PROMPT,        
    FULL_REJECTION_PROMPT
)

from app.core.security import verify_supabase_jwt

from src.f_prompts.skills import load_skill, list_skills

# In ra model Ollama đang dùng
print("Ollama model:", config.llm.ollama.available)
print("Ollama base_url:", config.llm.ollama.base_url)


Ollama model: ['gemma4:e2b']
Ollama base_url: http://localhost:11434/v1


In [3]:
available_tools = {
    "product_search": product_search,
    "product_compare": product_compare,
    "policy_search": policy_search,
    "order_lookup": order_lookup
}

tools_schema = [
    PRODUCT_SEARCH_SCHEMA,
    PRODUCT_COMPARE_SCHEMA,
    POLICY_SEARCH_SCHEMA,
    ORDER_LOOKUP_SCHEMA
]


In [4]:
def select_skill(query: str) -> str:
    """Chọn skill markdown phù hợp với câu hỏi. Có thể thay bằng LLM-call nhẹ sau này."""
    q = (query or "").lower()
    # Order / account
    if any(k in q for k in ["đơn hàng", "order", "tra đơn", "mua hàng"]):
        return load_skill("account/order_lookup") or ""
    # Policy
    if any(k in q for k in ["chính sách", "đổi trả", "bảo hành", "trả góp", "giao hàng", "vận chuyển"]):
        return load_skill("policy/policy_search") or ""
    # Compare (named products)
    if any(k in q for k in ["so sánh", "nên chọn", "hay hơn"]):
        return load_skill("product/compare") or load_skill("product/single_spec") or ""
    # Ambiguous / vague
    if any(k in q for k in ["tư vấn", "gợi ý", "nên mua", "máy nào", "con nào", "tầm giá"]):
        return load_skill("product/ambiguous") or ""
    # Single specific spec / price / stock
    if any(k in q for k in ["bao nhiêu", "giá", "tồn kho", "chip", "ram", "pin", "màn hình", "camera", "bộ nhớ"]):
        return load_skill("product/single_spec") or ""
    # Default: ambiguous / general guidance
    return load_skill("product/ambiguous") or ""

print("Available skills:", list_skills())


Available skills: ['account\\order_lookup', 'common\\multi_turn_context', 'policy\\policy_search', 'product\\ambiguous', 'product\\single_spec']


In [5]:
class MasterAgent:
    def __init__(self, llm_service: LLMService, config):
        self.llm_service = llm_service
        # Dùng model Ollama local (gemma4:2b) thay vì Gemini
        self.model = config.llm.ollama.available[0]

        # Danh sách tool cần authentication - chỉ cần tên tool
        # Thêm tool mới: chỉ append vào list, KHÔNG sửa logic invoke()
        self.AUTH_TOOLS = ["order_lookup", "cart_lookup", "wishlist_update"]

    # ==================================================================
    # HELPER 1: Chuẩn hóa đối số tool trước khi gọi hàm thật
    # ==================================================================
    def _sanitize_tool_args(self, name, args):
        """
        [CHỨC NĂNG]
        Chuẩn hóa và chỉ giữ các key hợp lệ trong tool args mà LLM trả về,
        trước khi truyền vào hàm Python thật.

        LÝ DO CẦN THIẾT:
        - LLM có thể trả về args ở nhiều định dạng khác nhau: chuỗi JSON/YAML,
          object phẳng (flat dict), list queries, hoặc dict lồng nhau. Hàm này
          chuẩn hóa mọi thứ về đúng cấu trúc dự kiến của từng tool.
        - Đảm bảo mỗi tool chỉ nhận đúng tập key hợp lệ (tránh key rác gây lỗi).
        """
        def _try_parse_object_string(s):
            """Thử parse chuỗi dạng { ... } thành dict. Không bắt buộc PyYAML."""
            if not isinstance(s, str):
                return None
            s = s.strip()
            if len(s) < 2 or not (s[0] == '{' and s[-1] == '}'):
                return None
            # 1. JSON chuẩn
            try:
                import json
                parsed = json.loads(s)
                if isinstance(parsed, dict):
                    return parsed
            except Exception:
                pass
            # 2. YAML nếu có
            try:
                import yaml
                parsed = yaml.safe_load(s)
                if isinstance(parsed, dict):
                    return parsed
            except Exception:
                pass
            # 3. Fallback: thêm dấu ngoặc kép cho key không có dấu ngoặc, rồi json.loads
            try:
                import json
                import re as _re
                normalized = _re.sub(r'([a-zA-Z_]\w*)\s*:', r'"\1":', s)
                parsed = json.loads(normalized)
                if isinstance(parsed, dict):
                    return parsed
            except Exception:
                pass
            return None

        # ===================================================================
        # Nhánh 1: product_search - xử lý cấu trúc queries phức tạp nhất
        # ===================================================================
        if name == "product_search":
            allowed_query_keys = {"keyword", "brand", "category", "min_price", "max_price", "name_contains", "mode", "limit", "include_details", "need_price_info"}

            # Args top-level có thể là string object, list queries, hoặc dict
            if isinstance(args, str):
                parsed = _try_parse_object_string(args)
                if isinstance(parsed, dict):
                    args = parsed
                else:
                    args = {"queries": [args]}
            elif isinstance(args, list):
                args = {"queries": args}
            elif not isinstance(args, dict):
                args = {}

            # Giá trị mặc định từ top-level (để merge vào các query thiếu)
            top_defaults = {k: args[k] for k in allowed_query_keys if k in args}
            top_limit = args.get("limit")
            default_limit = top_limit if top_limit is not None else 3

            # Nếu LLM gọi dạng flat (không có queries, keyword ở top-level) hoặc dùng 'query'
            if "queries" not in args:
                if "query" in args:
                    args = {"queries": [args["query"]]}
                elif "keyword" in args:
                    args = {"queries": [top_defaults]}
                else:
                    args = {"queries": []}

            queries = args.get("queries") or []
            if isinstance(queries, str):
                queries = [queries]

            clean_queries = []
            if isinstance(queries, list):
                for q in queries:
                    # Nếu q là string object, parse trước
                    if isinstance(q, str):
                        parsed_q = _try_parse_object_string(q)
                        if isinstance(parsed_q, dict):
                            q = parsed_q
                        else:
                            q = {"keyword": q}

                    if isinstance(q, dict):
                        # Nếu keyword là object-string (model copy JSON vào keyword), parse và merge
                        merged = top_defaults.copy()
                        kw = q.get("keyword")
                        parsed_kw = _try_parse_object_string(kw) if isinstance(kw, str) else None
                        if isinstance(parsed_kw, dict):
                            merged.update(parsed_kw)
                            for k, v in q.items():
                                if k != "keyword":
                                    merged[k] = v
                        else:
                            merged.update(q)

                        if not merged.get("keyword"):
                            merged["keyword"] = f"{merged.get('brand','')} {merged.get('category','')}".strip() or "sản phẩm"
                        
                        # Tránh name_contains quá ngắn/gây nhiễu (vd chỉ 'S')
                        nc = merged.get("name_contains")
                        if nc is not None and len(str(nc).strip()) <= 2:
                            merged["name_contains"] = merged.get("keyword")
                        
                        if "limit" not in merged or merged.get("limit") is None:
                            if merged.get("mode") == "lines":
                                merged["limit"] = 30
                            else:
                                merged["limit"] = default_limit
                        clean = {k: v for k, v in merged.items() if k in allowed_query_keys}
                        clean_queries.append(clean)

            result = {"queries": clean_queries}
            if top_limit is not None:
                result["limit"] = top_limit
            return result

        # ===================================================================
        # Nhánh 2: product_compare - chỉ giữ danh sách tên sản phẩm
        # ===================================================================
        if name == "product_compare":
            if not isinstance(args, dict):
                args = {}
            product_names = args.get("product_names") or args.get("products") or args.get("product_name")
            if isinstance(product_names, str):
                product_names = [product_names]
            if not isinstance(product_names, list):
                product_names = []
            return {"product_names": [p for p in product_names if isinstance(p, str)]}

        # ===================================================================
        # Nhánh 3: policy_search - chỉ giữ từ khóa và giới hạn kết quả
        # ===================================================================
        if name == "policy_search":
            if not isinstance(args, dict):
                args = {}
            return {k: v for k, v in args.items() if k in {"key_word", "limit"}}

        # ===================================================================
        # Nhánh 4: order_lookup - chỉ giữ order_id (auth được inject riêng)
        # ===================================================================
        if name == "order_lookup":
            if not isinstance(args, dict):
                args = {}
            return {k: v for k, v in args.items() if k in {"order_id"}}

        return args if isinstance(args, dict) else {}

    # ==================================================================
    # HELPER 2: Vòng lặp ReAct chính (Reasoning + Acting) - Ollama
    # ==================================================================
    def invoke(
        self, 
        messages: List[Dict[str, Any]],
        available_tools: Dict[str, Any] = None,
        tools_schema: List[Dict[str, Any]] = None,
        auth_context: dict = None,
        skill: str = None
        ):
        """
        [CHỨC NĂNG]
        Chạy vòng lặp ReAct bằng Ollama (OpenAI-compatible): gọi model local,
        nếu LLM yêu cầu gọi tool thì thực thi tool thật, đưa kết quả trở lại
        làm ngữ cảnh, lặp lại cho đến khi LLM trả lời cuối cùng hoặc hết lượt.
        """

        # Inject skill as an extra system message (right after the first system prompt)
        if skill:
            messages = list(messages)
            insert_at = 0
            for i, m in enumerate(messages):
                if isinstance(m, dict) and m.get("role") == "system":
                    insert_at = i + 1
                    break
            messages.insert(insert_at, {"role": "system", "content": skill})

        start_time = time.time()
        total_input_tokens = 0
        total_output_tokens = 0
        # Lưu lịch sử tool calls (args + output) để master_node inject vào lượt sau
        tool_context = []

        max_turns = config.agent.max_turns
        for turn in range(max_turns):
            turn_start_time = time.time()

            # ============================================================
            print(f"🌀 --- LƯỢT {turn + 1} (OLLAMA) ---")    
            # ============================================================

            # Gọi Ollama (OpenAI-compatible) - không stream, không retry phức tạp
            response = self.llm_service.call_ollama(
                model=self.model,
                messages=messages,
                tools=tools_schema,
                stream=False,
            )

            # Parse phản hồi OpenAI-format:
            #   response.choices[0].message.content      -> text
            #   response.choices[0].message.tool_calls    -> [{ id, function: {name, arguments} }]
            #   response.usage.prompt_tokens / completion_tokens -> token counts
            message = response.choices[0].message
            text_content = message.content or ""

            if response.usage:
                total_input_tokens += response.usage.prompt_tokens or 0
                total_output_tokens += response.usage.completion_tokens or 0

            turn_elapsed = time.time() - turn_start_time
            print(f"\n\n⏱️ [TURN {turn + 1} LATENCY]: {turn_elapsed:.2f}s")
            print(f"📊 [TURN {turn + 1} TOKENS]: Input = {response.usage.prompt_tokens if response.usage else 0} | Output = {response.usage.completion_tokens if response.usage else 0}")
            print(f"📝 [TURN {turn + 1} TEXT]: {text_content[:200] if text_content else '(tool call)'}")

            # Chuyển tool_calls của Ollama sang chuẩn OpenAI để lưu history
            formatted_tool_calls = []
            if message.tool_calls:
                for tc in message.tool_calls:
                    formatted_tool_calls.append({
                        "id": tc.id or f"call_ollama_{turn}_{len(formatted_tool_calls)}",
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments if isinstance(tc.function.arguments, str) else json.dumps(tc.function.arguments),
                        }
                    })

            # Lưu message của assistant vào history
            agent_msg = {
                "role": "assistant",
                "content": text_content if text_content else None
            }
            if formatted_tool_calls:
                agent_msg["tool_calls"] = formatted_tool_calls
                
            messages.append(agent_msg)
            
            # Thực thi Tools nếu LLM yêu cầu
            if formatted_tool_calls:
                print("\n🔧 LLM yêu cầu gọi Tool...")
                tool_start_time = time.time()
                
                for tc in formatted_tool_calls:
                    func_name = tc["function"]["name"]
                    func_args = json.loads(tc["function"]["arguments"]) if tc["function"]["arguments"] else {}
                    func_args = self._sanitize_tool_args(func_name, func_args)
                    
                    # Inject auth cho tool trong danh sách AUTH_TOOLS
                    if func_name in self.AUTH_TOOLS and auth_context:
                        func_args["current_user_id"] = auth_context.get("user_id")
                        func_args["user_token"] = auth_context.get("user_token")
                        print(f"   🔑 Injected auth for {func_name}")

                    if func_name in available_tools:
                        real_function = available_tools[func_name]
                        print(f"   👉 Chạy hàm: {func_name}({func_args})")

                        result = real_function(**func_args)
                        print(f"   📊 Kết quả từ Tool: {str(result)[:200]}")

                        # Lưu lại args + output vào tool_context để truyền sang lượt sau
                        tool_context.append({
                            "tool": func_name,
                            "args": self._sanitize_tool_args(func_name, func_args),
                            "output": str(result),
                        })

                        messages.append({
                            "role": "tool",
                            "tool_call_id": tc["id"],
                            "name": func_name,
                            "content": str(result)
                        })
                    else:
                        print(f"   ❌ Lỗi: Không tìm thấy tool '{func_name}'")

                tool_elapsed = time.time() - tool_start_time
                print(f"⏱️ [TOOL EXECUTION TIME]: {tool_elapsed:.2f}s")
                print("🔄 Gửi kết quả Tool lại cho Ollama suy luận tiếp...\n")
                continue
            else:
                total_elapsed = time.time() - start_time
                print("\n==================================================")
                print(f"✅ --- HOÀN THÀNH HOÀN TOÀN ---")
                print(f"⏱️ [TOTAL AGENT LATENCY]: {total_elapsed:.2f}s")
                print(f"📊 [TOTAL AGENT TOKENS]: Input = {total_input_tokens} | Output = {total_output_tokens} | Grand Total = {total_input_tokens + total_output_tokens}")
                print("==================================================\n")
                
                return {
                    "content": text_content if text_content else None,
                    "tool_context": tool_context,
                    "latency": total_elapsed,
                    "tokens": {
                        "input": total_input_tokens,
                        "output": total_output_tokens,
                    }
                }

In [6]:
llm_service = LLMService(settings, config)
guardrail_call = GuardrailCall(llm_service, config)
rejection_call = RejectionCall(llm_service, config)
master_agent = MasterAgent(llm_service, config)

print(f"🔑 Ollama model: {master_agent.model}")
print(f"🔑 LLMService has call_ollama: {hasattr(llm_service, 'call_ollama')}")


🔑 Ollama model: gemma4:e2b
🔑 LLMService has call_ollama: True


In [7]:
class RetrievedChunk(TypedDict):
    content: str
    source: str
    score: float
    chunk_type: str

class AgentState(TypedDict):

    # 1. INput user
    user_query: str
    session_id: str

    # 2. Auth
    user_token: Optional[str]
    user_id: Optional[str]
    is_authenticated: bool

    # 3. Guardrail & Quality
    risk_level: Optional[str]               # "low" | "medium" | "high"
    relevance_score: float                   # Dùng cho self-check Corrective RAG

    # 4. Router (Định tuyến)
    intent: str                              # Kết quả phân loại: 'product', 'policy', 'account', 'support'
    selected_agent: Optional[str]            # Quyết định Nút xử lý tiếp theo

    # 5. Retrieval & Tools
    retrieved_context: list[RetrievedChunk]
    tool_calls_used: Annotated[list[dict], operator.add]  # ⚡ Reducer cộng dồn lịch sử tool calls
    iteration_count: int                     # Đếm số lần lặp chống infinite loop

    # 6. Hội thoại
    messages: Annotated[list, add_messages]  # ⚡ Reducer cộng dồn tin nhắn
    conversation_state: dict

    # 7. Output
    final_answer: Optional[str]
    cited_sources: list[str]
    ticket_id: Optional[str]
    show_popup: bool

    # 8. Thống kê
    input_tokens: Annotated[int, operator.add]
    output_tokens: Annotated[int, operator.add]
    latency: Annotated[float, operator.add]
    total_tokens: Annotated[int, operator.add]

In [8]:
def receive_node(state: AgentState) -> dict:
    query = state.get("user_query", "").strip()
    token = state.get("user_token")
    user_id = None

    if token:
        user_id = verify_supabase_jwt(token)
    
    is_authenticated = True if user_id else False

    # ============================================================
    if is_authenticated:
        print(f"Người dùng đã xác thực")
    else:
        print(f"Người dùng chưa xác thực")
    # ============================================================
    
    return {
        "user_id": user_id,
        "is_authenticated": is_authenticated,
        "user_query": query  
    }

In [9]:
def guardrail_node(state: AgentState) -> dict:
    query = state["user_query"]
    
    result = guardrail_call.invoke(query)
    risk_level = result["risk_level"]
    
    # Chỉ lưu tin nhắn khi KHÔNG phải attack
    if risk_level != "attack":
        return {
            "risk_level": risk_level,
            "show_popup": False,
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ],
            "latency": result["latency"],
            "input_tokens": result["tokens"]["input"],
            "output_tokens": result["tokens"]["output"],
            "total_tokens": result["tokens"]["input"] + result["tokens"]["output"]
        }
    else:
        # Attack → KHÔNG lưu
        return {
            "risk_level": risk_level,
            "show_popup": True,
            "latency": result["latency"],
            "input_tokens": result["tokens"]["input"],
            "output_tokens": result["tokens"]["output"],
            "total_tokens": result["tokens"]["input"] + result["tokens"]["output"]
        }

In [10]:
def rejection_node(state: AgentState) -> dict:
    """
    [NODE] Rejection Agent (Từ chối):
    - Chỉ chạy khi risk_level == "needs_ticket"
    - Trả về câu từ chối lịch sự
    - KHÔNG lưu tin nhắn vào messages (đã lưu ở guardrail_node)
    """
    
    query = state["user_query"]

    result = rejection_call.invoke(query)

    return {
        "final_answer": result["content"],
        "show_popup": True,
        "input_tokens": result["tokens"]["input"],
        "output_tokens": result["tokens"]["output"],
        "total_tokens": result["tokens"]["input"] + result["tokens"]["output"],
        "latency": result["latency"]
    }

In [11]:
def master_node(state: AgentState) -> dict:
    """
    [NODE 3] Master Agent (Tư duy) - dùng Ollama:
    - Chỉ chạy khi risk_level != "attack"
    - Build messages: system prompt + lịch sử tool calls được inject giữa các lượt hội thoại
    """
    import json

    def _fmt_search_context(record):
        """Tóm tắt bộ lọc product_search đã dùng ở lượt trước."""
        if not record or record.get("tool") != "product_search":
            return ""
        args = record.get("args", {})
        queries = args.get("queries", [])
        if not queries:
            return ""
        q = queries[0]
        out = str(record.get("output", ""))
        n_items = 0
        if "Dòng:" in out:
            n_items = out.count("Dòng:")
        elif "Product:" in out:
            n_items = out.count("Product:")
        summary = f"{n_items} dòng" if "Dòng:" in out else (f"{n_items} sản phẩm" if "Product:" in out else "kết quả")
        skill = load_skill("common/multi_turn_context") or ""
        return skill.format(
            previous_query=json.dumps(q, ensure_ascii=False),
            result_summary=summary
        )

    tool_history = state.get("tool_calls_used") or []
    last_search_record = None
    for rec in reversed(tool_history):
        if rec.get("tool") == "product_search":
            last_search_record = rec
            break

    full_messages = [{"role": "system", "content": FULL_MASTER_PROMPT}]
    user_count = 0
    for msg in state["messages"]:
        role = msg.get("role") if isinstance(msg, dict) else getattr(msg, "type", "")
        if role == "user":
            user_count += 1
            if user_count > 1 and last_search_record:
                note = _fmt_search_context(last_search_record)
                if note:
                    full_messages.append({"role": "system", "content": note})
        full_messages.append(msg)

    auth_context = {
        "user_id": state.get("user_id"),
        "user_token": state.get("user_token")
    }

    user_query = state.get("user_query", "")
    skill_text = select_skill(user_query)
    if skill_text:
        print(f"🧩 Injected skill for query: {user_query[:60]}...")

    result = master_agent.invoke(
        messages=full_messages,
        available_tools=available_tools,
        tools_schema=tools_schema,
        auth_context=auth_context,
        skill=skill_text
    )

    new_tool_records = result.get("tool_context") or []

    assistant_msg = {
        "role": "assistant",
        "content": result["content"]
    }

    conversation_state = state.get("conversation_state") or {}
    found_new_search = False
    for rec in reversed(new_tool_records):
        if rec.get("tool") == "product_search":
            conversation_state["last_product_search"] = rec
            found_new_search = True
            break
    if not found_new_search and last_search_record:
        conversation_state["last_product_search"] = last_search_record

    return {
        "final_answer": result["content"],
        "messages": [assistant_msg],
        "tool_calls_used": new_tool_records,
        "conversation_state": conversation_state,
        "input_tokens": result["tokens"]["input"],
        "output_tokens": result["tokens"]["output"],
        "total_tokens": result["tokens"]["input"] + result["tokens"]["output"],
        "latency": result["latency"]
    }

In [12]:
builder = StateGraph(AgentState)

builder.add_node("receive_node", receive_node)
builder.add_node("guardrail_node", guardrail_node)
builder.add_node("rejection_node", rejection_node)
builder.add_node("master_node", master_node)

def route_after_guardrail(state: AgentState) -> str:
    """Hàm quyết định Nút tiếp theo dựa vào kết quả của Guardrail"""
    risk = state.get("risk_level", "safe")
    
    if risk == "attack":
        return "rejection_node"
    else:
        return "master_node"


builder.add_edge(START, "receive_node")
builder.add_edge("receive_node", "guardrail_node")
builder.add_conditional_edges(
    "guardrail_node",
    route_after_guardrail,
    {
        "rejection_node": "rejection_node", 
        "master_node": "master_node"       
    }
)
builder.add_edge("rejection_node", END)
builder.add_edge("master_node", END)

memory = MemorySaver()
app = builder.compile(checkpointer=memory)
print("🎉 Đã dựng thành công Đồ thị LangGraph Workflow 1 - Ollama!")

🎉 Đã dựng thành công Đồ thị LangGraph Workflow 1 - Ollama!


## Test: Chạy thử với Ollama (Gemma 4 2B)

> ⚠️ **Lưu ý**: Guardrail và Rejection vẫn dùng Gemini (`GuardrailCall`/`RejectionCall` gọi `call_gemini`).
> Chỉ Master Agent dùng Ollama. Nếu muốn chạy hoàn toàn local, cần sửa thêm guardrail/rejection sau.

In [16]:
user_token = ""  # Thay bằng JWT token nếu cần test order_lookup / auth

run_config = {"configurable": {"thread_id": "session_ollama_test_001"}}

# 1. Gọi Đồ thị chạy
res = app.invoke(
    {"user_query": "Chị muốn laptop hp e,có dòng nào phù hợp cho chị, chị cần cho con máy làm đồ họa",
     "user_token": user_token}, 
    config=run_config
)

# 2. IN BÁO CÁO THỐNG KÊ CHI TIẾT TỪ AGENT STATE
print()
print("📊 BÁO CÁO THỐNG KÊ CHI TIẾT (AGENT STATE METRICS)")
print("="*60)
print(f"💬 Câu trả lời (Final Answer) : {res.get('final_answer')}")
print(f"🛡️ Mức độ rủi ro (Risk Level) : {res.get('risk_level')}")
print(f"⏱️ Tổng độ trễ (Total Latency): {res.get('latency', 0):.2f}s")
print(f"📥 Input Tokens               : {res.get('input_tokens', 0)}")
print(f"📤 Output Tokens              : {res.get('output_tokens', 0)}")
print(f"🧮 Tổng Tokens (Total Tokens)  : {res.get('total_tokens', 0)}")

# 3. IN LỊCH SỬ HỘI THOẠI TRONG MEMORY
print("\n📜 LỊCH SỬ HỘI THOẠI (MESSAGES HISTORY):")
for idx, msg in enumerate(res.get("messages", []), 1):
    role = getattr(msg, "type", None) or (msg.get("role") if isinstance(msg, dict) else "unknown")
    content = getattr(msg, "content", None) or (msg.get("content") if isinstance(msg, dict) else "")
    if isinstance(content, str) and len(content) > 300:
        content = content[:300] + "..."
    print(f"  [{idx}] {role.upper()}: {content}")

print("="*60)


Người dùng chưa xác thực
🧩 Injected skill for query: Chị muốn laptop hp e,có dòng nào phù hợp cho chị, chị cần ch...
🌀 --- LƯỢT 1 (OLLAMA) ---


⏱️ [TURN 1 LATENCY]: 25.63s
📊 [TURN 1 TOKENS]: Input = 5610 | Output = 512
📝 [TURN 1 TEXT]: (tool call)

🔧 LLM yêu cầu gọi Tool...
   👉 Chạy hàm: product_search({'queries': [{'keyword': 'laptop đồ họa', 'brand': 'HP', 'max_price': 30000000, 'mode': 'lines', 'category': 'laptop', 'include_details': True, 'limit': 30}]})
   📊 Kết quả từ Tool: ["laptop đồ họa"] - 23 dòng máy:
Dòng: Laptop HP Pavilion
Đại diện: Laptop HP Pavilion 14-CE3013TU 8QN72PA
Giá: 11,990,000 VND | Tình trạng: Hết hàng | SKU: LAP-HP-PAV14-CE3013TU-8QN72PA CORE I3-1005G
⏱️ [TOOL EXECUTION TIME]: 4.39s
🔄 Gửi kết quả Tool lại cho Ollama suy luận tiếp...

🌀 --- LƯỢT 2 (OLLAMA) ---


⏱️ [TURN 2 LATENCY]: 58.72s
📊 [TURN 2 TOKENS]: Input = 9812 | Output = 628
📝 [TURN 2 TEXT]: Dạ em tìm được một số mẫu laptop HP trong tầm giá dưới 30 triệu mà anh/chị có thể tham khảo ạ.

Tuy nhiên, 

### Kiểm tra nhanh call_ollama độc lập

Nếu graph chạy lỗi, có thể test trực tiếp `call_ollama` trước:

In [14]:
# # Test độc lập call_ollama (không qua graph)
# try:
#     resp = llm_service.call_ollama(
#         model=config.llm.ollama.available[0],
#         messages=[{"role": "user", "content": "Xin chào, bạn là ai?"}],
#         stream=False,
#     )
#     print("✅ Ollama trả lời:", resp.choices[0].message.content)
# except Exception as e:
#     print(f"❌ Lỗi Ollama: {e}")
